# Preprocessing

The text content is not in its correct format - it is currently in a word format. This data cannot be trained on, so we must put it through a process called tokenisation to convert it into a format the model will understand (this will be explained throughout the notebooks). 

Of course, tokenisation is not possible without first cleaning up the data. 

## Imports

Firstly, let's import our data as a CSV. 

In [1]:
import pandas as pd

df = pd.read_parquet('csv/all_data.parquet')

In this section, I am using a library called NLTK (Natural Language Toolkit), which contains utilities for being able to convert sentences into tokens.   

In [2]:
import nltk
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
import re
import contractions
from joblib import Parallel, delayed # makes it faster to apply functions

n_jobs = 2 # core count to use when doing parallel threading

This downloads samples and datasets for the nltk to use. I'm not particularly sure about what it does, NLTK functions throw an error asking for this to be downloaded. 

In [3]:
nltk.download('stopwords')
nltk.download('twitter_samples')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package twitter_samples to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Stopwords

Stopwords are words that carry semantic meaning and can make analysis of the words hard. It includes such articles ("a", "the", ...), conjunctions ("and", "or", ...), prepositions ("in", "on", ...) and other such words. 

By removing stopwords, we are able to remove any tokens/words that do not hold minimal meaning. 

In [4]:
stop_words = set(stopwords.words('english'))

There are some words that can drastically change the meaning if removed. 

For example:

"That is not good" (negative) ----removing stopword---> "That is good" (positive) *[not what we want]*

By excluding the negations from the stopwords, we will be able to keep the sentiment of the text block

In [5]:
negations = {'no', 'not', 'nor', 'neither', 'never', 'none',
            "don't", "won't", "can't", "isn't", "aren't", "wasn't"}
stop_words = stop_words - negations

## Tokenising

By tokenising, we are able to convert a long block of text into individual words. 

In [6]:
tokeniser = TweetTokenizer(strip_handles=True, reduce_len=True)

In [7]:
from joblib import Parallel, delayed
import numpy as np

def clean_text_content(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'&\w+;', '', text)
    text = contractions.fix(text)
    text = text.lower().strip()
    return text

def process_chunk(chunk):
    return chunk.apply(clean_text_content)

df = df.reset_index(drop=True)

chunks = [df['Text Content'].iloc[i::4] for i in range(4)]
results = Parallel(n_jobs=n_jobs)(delayed(process_chunk)(chunk) for chunk in chunks)
df['Mutated Text Content'] = pd.concat(results).sort_index().reset_index(drop=True)

df.head()

,Sentiment,Text Content,tweet_length,Mutated Text Content
0,irrelevant,I freaking knew @KaitlinPlacko @allietrezz and...,135,i freaking knew @kaitlinplacko @allietrezz and...
1,irrelevant,How Abhijeet Singh Bhayya ur efforts... are in...,217,how abhijeet singh bhayya you are efforts... a...
2,positive,"Thanks, @MSFTImagine & @IamPablo kindly sendin...",104,"thanks, @msftimagine & @iampablo kindly sendin..."
3,positive,Just bought GTA I’m done,24,just bought gta i am done
4,positive,UGH SO AWESOME!!!,17,ugh so awesome!!!


In [8]:
def tokenise_text_content(text):
    if not text:
        return []
    tokens = tokeniser.tokenize(text)
    tokens = [re.sub(r'[^\w\s]', '', w) for w in tokens]
    tokens = [w for w in tokens if w and w not in stop_words]
    return tokens

def process_token_chunk(chunk):
    return chunk.apply(tokenise_text_content)

df = df.reset_index(drop=True)

chunks = [df['Mutated Text Content'].iloc[i::4] for i in range(4)]
results = Parallel(n_jobs=n_jobs)(delayed(process_token_chunk)(chunk) for chunk in chunks)
df["Text Tokens"] = pd.concat(results).sort_index().reset_index(drop=True)

df = df[df["Text Tokens"].apply(len) > 0].reset_index(drop=True)

df.head()

,Sentiment,Text Content,tweet_length,Mutated Text Content,Text Tokens
0,irrelevant,I freaking knew @KaitlinPlacko @allietrezz and...,135,i freaking knew @kaitlinplacko @allietrezz and...,"[freaking, knew, cracked, fortnite, knew, coul..."
1,irrelevant,How Abhijeet Singh Bhayya ur efforts... are in...,217,how abhijeet singh bhayya you are efforts... a...,"[abhijeet, singh, bhayya, efforts, incredible,..."
2,positive,"Thanks, @MSFTImagine & @IamPablo kindly sendin...",104,"thanks, @msftimagine & @iampablo kindly sendin...","[thanks, kindly, sending, awesome, microsoft, ..."
3,positive,Just bought GTA I’m done,24,just bought gta i am done,"[bought, gta, done]"
4,positive,UGH SO AWESOME!!!,17,ugh so awesome!!!,"[ugh, awesome]"


## Classification of sentiment

As previously mentioned, there are ~~4~~ 3 primary types of sentiment classification: 
- Positive
- Neutral/Irrelevant
- Negative

Under initial testing, neutral/irrelevant data is not accurate to be used, as it skews the dataset (in terms of size), but also it is not a great method of determining sentiment. 

Within the set, we can use binary classification as so:
- Positive (1)
- Negative (0) (it only made sense if neutral was 0, not Negative)

Firstly, lets remove the sentiments that are not needed (Neutral/Irrelevant). 

In [9]:
df = df[~df['Sentiment'].isin(['neutral', 'irrelevant'])]

Now, we need to map the string content into a numerical form, something that the models can understand. 

In [10]:
s_map = {
    'positive': 1,
    'negative': 0,
}

df['Sentiment'] = df['Sentiment'].map(s_map)
df.head()

,Sentiment,Text Content,tweet_length,Mutated Text Content,Text Tokens
2,1,"Thanks, @MSFTImagine & @IamPablo kindly sendin...",104,"thanks, @msftimagine & @iampablo kindly sendin...","[thanks, kindly, sending, awesome, microsoft, ..."
3,1,Just bought GTA I’m done,24,just bought gta i am done,"[bought, gta, done]"
4,1,UGH SO AWESOME!!!,17,ugh so awesome!!!,"[ugh, awesome]"
5,1,It's no secret that I love Assassin's Creed. I...,175,it is no secret that i love assassin's creed. ...,"[no, secret, love, assassins, creed, extremely..."
6,1,WORLD LIVE IN TELUGU | BE MUCH FUN | BE LIVE,44,world live in telugu | be much fun | be live,"[world, live, telugu, much, fun, live]"


# Finish

Preprocessing is completed. Let's save into a .csv to be modified in feature_engineering

In [ ]:
import os
os.makedirs('csv', exist_ok=True)
df.to_parquet('csv/preprocess.parquet', index=False, engine='pyarrow', compression='snappy')